# Collatz, from the library's arrows

The rule $T(n) = n/2$ for even $n$, $3n+1$ for odd $n$, and the question whether every orbit reaches 1. This notebook builds the rule from the library's own arrows, iterates it, searches the orbit, and reads the search in K₃, so every answer is a value of a library primitive.

| here | primitive |
|---|---|
| `π`, `π % 2 == 0`, `π // 2`, `3*π + 1` | the identity arrow on ℕ's shadow; `Mod`, `FloorDiv`, `Affine` composed after it; `preimage(f, η(k))` for the test |
| `cond(p, f, g)` | McCarthy's conditional, `category::Cond` |
| `iterate(T, n)` | `sequences::iterate`, a lazy Path |
| `o.first_where(π == 1, B)` | `sequences::first_where`, the bounded search |
| `Σ` | `category::Semidecided`: a search read in K₃ |
| `forall(W, P)` | the meet of P along the window, in K₃ |

Each cell asserts what it shows.

In [ ]:
from dedekind.collatz import π, cond, iterate, Σ, reaches_within, forall, collatz_step, at_least, below, everything, Ternary
U, T = Ternary.UNKNOWN, Ternary.TRUE

step = cond(π % 2 == 0, π // 2, 3*π + 1)      # the rule, as a term
assert step(27) == 82 and step(6) == 3 and collatz_step(27) == 82   # collatz_step is the library's own term
step(27), step(6)

## The orbit is the iterate

`iterate(step, 27)` is the path $27, T(27), T^2(27), \ldots$, computed lazily. It does not end: once at 1 it cycles $1, 4, 2, 1$. The bounded search `first_where(π == 1, B)` returns the first index at which the path is 1, or `None` if none within the budget.

In [ ]:
o = iterate(step, 27)
assert o[:6] == [27, 82, 41, 124, 62, 31] and max(o[:112]) == 9232
assert o.first_where(π == 1, 120) == 111 and o.first_where(π == 1, 50) is None
o[:12], max(o[:112]), o.first_where(π == 1, 120)

## Σ reads a search in K₃

A bounded search has two outcomes: a witness, or none so far. `Σ` maps them to ⊤ and U. There is no ⊥, because an exhausted budget refutes nothing. `reaches_within(step, B)` is the composition $Σ \circ \mathrm{first\_where}(π = 1, B) \circ \mathrm{iterate}(step)$, packaged as a predicate on ℕ valued in K₃.

In [ ]:
assert Σ(True) == T and Σ(False) == U
P = reaches_within(step, 120)
assert P(27) == T and P(27) == Σ(o.first_where(π == 1, 120) is not None)
assert reaches_within(step, 50)(27) == U
[(B, reaches_within(step, B)(27)) for B in (50, 110, 111, 120)]

## A window with a budget decides

Over the window $[1, 1000)$ the ∀ is the meet of the predicate along the window, in K₃. It answers U until the budget covers the slowest seed in the window, 871 at 178 steps, and ⊤ from there. ⊤ means the predicate takes only the values ⊥ and ⊤ on the window: it has become a Boolean set, and the window is decided.

In [ ]:
W = at_least(1) & below(1000)
sweep = [(B, forall(W, reaches_within(step, B))) for B in (50, 100, 177, 178, 300)]
assert [v for _, v in sweep] == [U, U, U, T, T]
sweep

## Change a constant, change the computation

The rule is a term, so one token changes it. With $5n+1$ in the odd branch, most orbits run away (the saturating arithmetic parks them at the word's top), so the window ∀ stays U at any budget.

In [ ]:
five = cond(π % 2 == 0, π // 2, 5*π + 1)
assert iterate(five, 1)[:8] == [1, 6, 3, 16, 8, 4, 2, 1]
assert reaches_within(five, 20)(7) == U
assert forall(at_least(1) & below(100), reaches_within(five, 1000)) == U
iterate(five, 7)[:10]

## Without a window there is no ∀

The conjecture says every seed arrives within some budget, $\forall n\, \exists B$. The library performs only the swapped form, $\exists B\, \forall n < W$: a ray or 𝔸 has no window, and the ∀ over it is refused rather than attempted. (The generalised problem, with arbitrary affine branches, is undecidable: Conway, 1972.)

In [ ]:
try:
    forall(everything(), reaches_within(step, 300))
    raise AssertionError('the unbounded ∀ should be refused')
except TypeError as e:
    print(e)